# Gemini 3.1 Pro - Prompt Pilot for MSc Dissertation

Runs 3 prompt variants against a stratified pilot sample of 14 videos and produces a comparison table.

**Before running:**
1. Put your Gemini API key in Colab Secrets under the name `GEMINI_API_KEY` (left sidebar → key icon → Add new secret).
2. Mount Google Drive when prompted.
3. Edit `PILOT_VIDEOS` in Cell 4 to point to your 14 chosen videos.

Total cost estimate: ~£1–2 (42 API calls).


## 1. Install and import

In [1]:
!pip install -q google-genai
import os, json, time, pathlib, re
from datetime import datetime
from google import genai
from google.genai import types
from google.colab import userdata, drive


## 2. Auth and Drive

In [2]:
API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=API_KEY)
drive.mount('/content/drive')
print("Ready.")


Mounted at /content/drive
Ready.


## 3. Prompt variants (three)

In [3]:
PROMPT_A = "You are a forensic examiner analysing a short video clip for artefacts characteristic of AI-generated content (text-to-video models). Identify which artefact families, if any, are present, using the taxonomy below. The video may or may not be AI-generated; base your judgment on evidence in the video itself.\n\n# ARTEFACT TAXONOMY (12 families)\n\n## Domain 1: Surface Artefacts (per-frame visual failures)\n1.1 Texture defects \u2014 waxy skin, plastic-looking materials, repetitive backgrounds, loss of fine detail (fabric weave, hair strands, wood grain).\n1.2 Boundary defects \u2014 blurred or soft object edges, halo/bleed around subjects, chromatic fringing, subject-background dissolution.\n1.3 Lighting inconsistency \u2014 missing or wrong-direction shadows, impossible light sources, non-physical reflections, lighting mismatched to scene.\n1.4 Watermark / provenance signal \u2014 visible watermarks, logos, or overlay text indicating generator origin (e.g. \"Sora\", \"Veo\", \"Kling\"), OR conspicuous absence-of-noise patterns and frequency-domain regularities suggestive of synthesis. Judge only from what is visible in the pixels; do not infer from filename or context.\n\n## Domain 2: Structural Defects (object and scene structure)\n2.1 Human anatomy \u2014 face defects (eyes, teeth, proportions), hand defects (finger count, grip), body issues (extra/missing limbs, impossible joints).\n2.2 Non-human anatomy \u2014 wrong limb count on animals, distorted animal faces, impossible fur/feather/scale rendering. If no animals or non-human creatures appear, return detected: false with evidence \"no non-human subjects present\".\n2.3 Object structural \u2014 distorted mechanical objects, text rendering failures (gibberish signs), wrong scale relationships, impossible topology.\n2.4 Scene composition \u2014 impossible spatial arrangements, missing expected objects, wrong perspective.\n\n## Domain 3: Temporal-Semantic Violations (across-frame, motion-visible only)\n3.1 Motion artefacts \u2014 jittery stationary objects, non-rigid motion of rigid objects, foot sliding during walking, impossible acceleration.\n3.2 Identity and object drift \u2014 face morphing between frames, clothing pattern changing, colour shifting on same object, object count changing.\n3.3 Continuity errors \u2014 objects appearing/disappearing without cause, sudden lighting shifts within a shot, background motion inconsistent with foreground, loop/repeat motion.\n3.4 Causality and physics violations \u2014 irreversibility violation (spilled liquid returning), conservation of matter violation, gravity/momentum failures, cause-effect mismatch.\n\n# SEVERITY BANDS \u2014 use the full range\n\n- L (Low): artefact present but subtle. A viewer would need to pause or look closely to notice. Example: slight waxiness on cheek skin only visible on a still frame; minor edge softness on hair.\n- M (Medium): artefact clearly present at normal playback speed and noticeable to an attentive viewer, but not the dominant feature of the frame. Example: one finger visibly merged with an adjacent finger; background text partially unreadable; a shadow direction that seems off but not impossible.\n- H (High): artefact is obvious, dominant, and would be immediately visible to a casual viewer. Example: hand with six fingers or fingers melting into each other; a person's face morphing shape mid-shot; an object phasing through a solid surface; text that is complete gibberish across the whole sign.\n\nDo NOT default to M. If the artefact is subtle, use L. If it is dominant and unmistakable, use H. A response where every detected family is \"M\" is almost certainly miscalibrated \u2014 reconsider.\n\n# OUTPUT FORMAT\n\nRespond ONLY with the JSON below. No preamble, no code fences, no trailing commentary.\n\n{\n  \"video_verdict\": \"AI-generated\" | \"Real\" | \"Uncertain\",\n  \"confidence\": <integer 0-100>,\n  \"families\": {\n    \"1.1_texture\":          {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"<observation with timestamp 0:XX>\"},\n    \"1.2_boundary\":         {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"1.3_lighting\":         {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"1.4_watermark\":        {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.1_human_anatomy\":    {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.2_non_human_anatomy\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.3_object_structural\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.4_scene_composition\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.1_motion\":           {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.2_identity_drift\":   {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.3_continuity\":       {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.4_causality\":        {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"}\n  },\n  \"most_diagnostic_artefact\": \"<one sentence naming the single most decisive observation, or 'no strong artefacts observed'>\",\n  \"notes\": \"<one to two sentences of additional observation, or empty string>\"\n}\n\n# RULES\n\n1. Mark \"detected\": true only when you observe specific evidence in the video, not on suspicion.\n2. When \"detected\" is false, set \"severity\": \"none\" and \"evidence\" to a brief reason.\n3. Every \"evidence\" field for a detected artefact must include a timestamp 0:XX.\n4. \"Real\" is a valid verdict. Do not assume AI-generation.\n5. Use the full L/M/H range as calibrated above.\n6. Output must be valid JSON, all 12 family keys present, no additional keys, no markdown."

PROMPT_B = "You are a forensic examiner analysing a short video clip for artefacts characteristic of AI-generated content (text-to-video models). Identify which artefact families, if any, are present, using the taxonomy below. The video may or may not be AI-generated; base your judgment on evidence in the video itself.\n\n# ARTEFACT TAXONOMY (12 families)\n\n## Domain 1: Surface Artefacts (per-frame visual failures)\n1.1 Texture defects \u2014 waxy skin, plastic-looking materials, repetitive backgrounds, loss of fine detail (fabric weave, hair strands, wood grain).\n1.2 Boundary defects \u2014 blurred or soft object edges, halo/bleed around subjects, chromatic fringing, subject-background dissolution.\n1.3 Lighting inconsistency \u2014 missing or wrong-direction shadows, impossible light sources, non-physical reflections, lighting mismatched to scene.\n1.4 Watermark / provenance signal \u2014 visible watermarks, logos, or overlay text indicating generator origin (e.g. \"Sora\", \"Veo\", \"Kling\"), OR conspicuous absence-of-noise patterns and frequency-domain regularities suggestive of synthesis. Judge only from what is visible in the pixels; do not infer from filename or context.\n\n## Domain 2: Structural Defects (object and scene structure)\n2.1 Human anatomy \u2014 face defects (eyes, teeth, proportions), hand defects (finger count, grip), body issues (extra/missing limbs, impossible joints).\n2.2 Non-human anatomy \u2014 wrong limb count on animals, distorted animal faces, impossible fur/feather/scale rendering. If no animals or non-human creatures appear, return detected: false with evidence \"no non-human subjects present\".\n2.3 Object structural \u2014 distorted mechanical objects, text rendering failures (gibberish signs), wrong scale relationships, impossible topology.\n2.4 Scene composition \u2014 impossible spatial arrangements, missing expected objects, wrong perspective.\n\n## Domain 3: Temporal-Semantic Violations (across-frame, motion-visible only)\n3.1 Motion artefacts \u2014 jittery stationary objects, non-rigid motion of rigid objects, foot sliding during walking, impossible acceleration.\n3.2 Identity and object drift \u2014 face morphing between frames, clothing pattern changing, colour shifting on same object, object count changing.\n3.3 Continuity errors \u2014 objects appearing/disappearing without cause, sudden lighting shifts within a shot, background motion inconsistent with foreground, loop/repeat motion.\n3.4 Causality and physics violations \u2014 irreversibility violation (spilled liquid returning), conservation of matter violation, gravity/momentum failures, cause-effect mismatch.\n\n# SEVERITY BANDS \u2014 use the full range\n\n- L (Low): artefact present but subtle. A viewer would need to pause or look closely to notice. Example: slight waxiness on cheek skin only visible on a still frame; minor edge softness on hair.\n- M (Medium): artefact clearly present at normal playback speed and noticeable to an attentive viewer, but not the dominant feature of the frame. Example: one finger visibly merged with an adjacent finger; background text partially unreadable; a shadow direction that seems off but not impossible.\n- H (High): artefact is obvious, dominant, and would be immediately visible to a casual viewer. Example: hand with six fingers or fingers melting into each other; a person's face morphing shape mid-shot; an object phasing through a solid surface; text that is complete gibberish across the whole sign.\n\n# CONFIDENCE CALIBRATION (integer 0\u2013100)\n\nUse the full range. Anchors:\n- 0\u201320: I am confident this video is Real. No credible AI markers.\n- 30\u201350: Genuinely uncertain. Some markers either way but nothing decisive.\n- 60\u201380: Probably AI-generated. Multiple clear markers but some ambiguity.\n- 85\u2013100: Almost certainly AI-generated. Multiple severe artefacts or a watermark.\n\nA batch of responses all clustered at 60\u201375 indicates you are hedging. Use the extremes when the evidence warrants.\n\nDo NOT default to M. If the artefact is subtle, use L. If it is dominant and unmistakable, use H. A response where every detected family is \"M\" is almost certainly miscalibrated \u2014 reconsider.\n\n# OUTPUT FORMAT\n\nRespond ONLY with the JSON below. No preamble, no code fences, no trailing commentary.\n\n{\n  \"video_verdict\": \"AI-generated\" | \"Real\" | \"Uncertain\",\n  \"confidence\": <integer 0-100>,\n  \"families\": {\n    \"1.1_texture\":          {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"<observation with timestamp 0:XX>\"},\n    \"1.2_boundary\":         {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"1.3_lighting\":         {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"1.4_watermark\":        {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.1_human_anatomy\":    {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.2_non_human_anatomy\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.3_object_structural\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.4_scene_composition\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.1_motion\":           {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.2_identity_drift\":   {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.3_continuity\":       {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.4_causality\":        {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"}\n  },\n  \"most_diagnostic_artefact\": \"<one sentence naming the single most decisive observation, or 'no strong artefacts observed'>\",\n  \"notes\": \"<one to two sentences of additional observation, or empty string>\"\n}\n\n# RULES\n\n1. Mark \"detected\": true only when you observe specific evidence in the video, not on suspicion.\n2. When \"detected\" is false, set \"severity\": \"none\" and \"evidence\" to a brief reason.\n3. Every \"evidence\" field for a detected artefact must include (a) a timestamp 0:XX and (b) a spatial location (e.g. \"bottom-left\", \"on the subject's right hand\", \"in the mirror reflection\", \"across the whole frame\").\n4. \"Real\" is a valid verdict. Do not assume AI-generation.\n5. Use the full L/M/H range as calibrated above.\n6. Output must be valid JSON, all 12 family keys present, no additional keys, no markdown."

PROMPT_C = "You are a forensic examiner analysing a short video clip for artefacts characteristic of AI-generated content (text-to-video models). Identify which artefact families, if any, are present, using the taxonomy below. The video may or may not be AI-generated; base your judgment on evidence in the video itself.\n\n# ARTEFACT TAXONOMY (12 families)\n\n## Domain 1: Surface Artefacts (per-frame visual failures)\n1.1 Texture defects \u2014 waxy skin, plastic-looking materials, repetitive backgrounds, loss of fine detail (fabric weave, hair strands, wood grain).\n1.2 Boundary defects \u2014 blurred or soft object edges, halo/bleed around subjects, chromatic fringing, subject-background dissolution.\n1.3 Lighting inconsistency \u2014 missing or wrong-direction shadows, impossible light sources, non-physical reflections, lighting mismatched to scene.\n1.4 Watermark / provenance signal \u2014 visible watermarks, logos, or overlay text indicating generator origin (e.g. \"Sora\", \"Veo\", \"Kling\"), OR conspicuous absence-of-noise patterns and frequency-domain regularities suggestive of synthesis. Judge only from what is visible in the pixels; do not infer from filename or context.\n\n## Domain 2: Structural Defects (object and scene structure)\n2.1 Human anatomy \u2014 face defects (eyes, teeth, proportions), hand defects (finger count, grip), body issues (extra/missing limbs, impossible joints).\n2.2 Non-human anatomy \u2014 wrong limb count on animals, distorted animal faces, impossible fur/feather/scale rendering. If no animals or non-human creatures appear, return detected: false with evidence \"no non-human subjects present\".\n2.3 Object structural \u2014 distorted mechanical objects, text rendering failures (gibberish signs), wrong scale relationships, impossible topology.\n2.4 Scene composition \u2014 impossible spatial arrangements, missing expected objects, wrong perspective.\n\n## Domain 3: Temporal-Semantic Violations (across-frame, motion-visible only)\n3.1 Motion artefacts \u2014 jittery stationary objects, non-rigid motion of rigid objects, foot sliding during walking, impossible acceleration.\n3.2 Identity and object drift \u2014 face morphing between frames, clothing pattern changing, colour shifting on same object, object count changing.\n3.3 Continuity errors \u2014 objects appearing/disappearing without cause, sudden lighting shifts within a shot, background motion inconsistent with foreground, loop/repeat motion.\n3.4 Causality and physics violations \u2014 irreversibility violation (spilled liquid returning), conservation of matter violation, gravity/momentum failures, cause-effect mismatch.\n\n# SEVERITY BANDS \u2014 use the full range\n\n- L (Low): artefact present but subtle. A viewer would need to pause or look closely to notice. Example: slight waxiness on cheek skin only visible on a still frame; minor edge softness on hair.\n- M (Medium): artefact clearly present at normal playback speed and noticeable to an attentive viewer, but not the dominant feature of the frame. Example: one finger visibly merged with an adjacent finger; background text partially unreadable; a shadow direction that seems off but not impossible.\n- H (High): artefact is obvious, dominant, and would be immediately visible to a casual viewer. Example: hand with six fingers or fingers melting into each other; a person's face morphing shape mid-shot; an object phasing through a solid surface; text that is complete gibberish across the whole sign.\n\nDo NOT default to M. If the artefact is subtle, use L. If it is dominant and unmistakable, use H. A response where every detected family is \"M\" is almost certainly miscalibrated \u2014 reconsider.\n\n# OUTPUT FORMAT\n\nFirst, write exactly 3 short sentences describing what you literally see in the video (subject, setting, motion). Prefix these with \"OBSERVATION:\" on its own line. Then leave a blank line. Then emit the JSON below.\n\nThe observation step must be grounded in the pixels, not in guesses about origin. Only after writing the observation should you decide whether artefacts are present.\n\nAfter the observation, respond with the JSON below. No preamble, no code fences, no trailing commentary.\n\n{\n  \"video_verdict\": \"AI-generated\" | \"Real\" | \"Uncertain\",\n  \"confidence\": <integer 0-100>,\n  \"families\": {\n    \"1.1_texture\":          {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"<observation with timestamp 0:XX>\"},\n    \"1.2_boundary\":         {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"1.3_lighting\":         {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"1.4_watermark\":        {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.1_human_anatomy\":    {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.2_non_human_anatomy\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.3_object_structural\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.4_scene_composition\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.1_motion\":           {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.2_identity_drift\":   {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.3_continuity\":       {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.4_causality\":        {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"}\n  },\n  \"most_diagnostic_artefact\": \"<one sentence naming the single most decisive observation, or 'no strong artefacts observed'>\",\n  \"notes\": \"<one to two sentences of additional observation, or empty string>\"\n}\n\n# RULES\n\n1. Mark \"detected\": true only when you observe specific evidence in the video, not on suspicion.\n2. When \"detected\" is false, set \"severity\": \"none\" and \"evidence\" to a brief reason.\n3. Every \"evidence\" field for a detected artefact must include a timestamp 0:XX.\n4. \"Real\" is a valid verdict. Do not assume AI-generation.\n5. Use the full L/M/H range as calibrated above.\n6. Output must be valid JSON, all 12 family keys present, no additional keys, no markdown."

PROMPTS = {"A_baseline": PROMPT_A, "B_calibrated": PROMPT_B, "C_observe_first": PROMPT_C}
for k, v in PROMPTS.items():
    print(f"{k}: {len(v)} chars, ~{len(v)//4} tokens")


A_baseline: 5730 chars, ~1432 tokens
B_calibrated: 6357 chars, ~1589 tokens
C_observe_first: 6131 chars, ~1532 tokens


## 4. Pilot sample

Point each entry at an actual video file in your Drive. Pick 2 per generator, ideally spanning quality (one clean-looking, one clearly broken). Include 2 real Pexels videos — critical for false-positive testing.


In [15]:
PILOT_ROOT = "/content/drive/MyDrive/msc-deepfake/generated_videos_final"

PILOT_VIDEOS = [
    # (generator_label, filename relative to PILOT_ROOT)
    ("LTX",         "ltx/w2_005_ltx_20260719_105736.mp4"),
    ("LTX",         "ltx/w2_020_ltx_20260719_110441.mp4"),
    ("Hunyuan",     "hunyuan/w2_003_hunyuan_20260719_133334.mp4"),
    ("Hunyuan",     "hunyuan/w2_015_hunyuan_20260719_141252.mp4"),
    ("Wan",         "wan/w2_007_wan_20260719_164920.mp4"),
    ("Wan",         "wan/w2_022_wan_20260719_175144.mp4"),
    ("Kling",       "kling/w2_004_kling.mp4"),
    ("Kling",       "kling/w2_018_kling.mp4"),
    ("Gemini_Omni", "gemini_omni/w2_006_gemini_omni_flash.mp4"),
    ("Gemini_Omni", "gemini_omni/w2_021_gemini_omni_flash.mp4"),
    ("Seedance",    "seedance/w2_008_seedance.mp4"),
    ("Seedance",    "seedance/w2_024_seedance.mp4"),
    ("Pexels",      "pexels/pexels_texture_28000739.mp4"),
    ("Pexels",      "pexels/pexels_motion_7187057.mp4"),
]
print(f"Pilot: {len(PILOT_VIDEOS)} videos × {3} prompts = {len(PILOT_VIDEOS)*3} calls")


Pilot: 14 videos × 3 prompts = 42 calls


## 5. Analyse one video with one prompt

In [16]:
MODEL = "gemini-3.1-pro-preview"
TEMPERATURE = 0.0

def analyse_video(video_path: str, system_prompt: str) -> dict:
    """Uploads a video, sends it to Gemini, returns the raw text response."""
    uploaded = client.files.upload(file=video_path)
    # Wait for the file to be ACTIVE
    while uploaded.state.name == "PROCESSING":
        time.sleep(2)
        uploaded = client.files.get(name=uploaded.name)
    if uploaded.state.name != "ACTIVE":
        raise RuntimeError(f"Upload failed: {uploaded.state.name}")

    response = client.models.generate_content(
        model=MODEL,
        contents=[uploaded, "Analyse this video per your system instruction."],
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=TEMPERATURE,
            response_mime_type="application/json",
        ),
    )
    # Clean up the uploaded file to avoid quota buildup
    try:
        client.files.delete(name=uploaded.name)
    except Exception:
        pass

    return {
        "raw_text": response.text,
        "usage": {
            "input_tokens": response.usage_metadata.prompt_token_count if response.usage_metadata else None,
            "output_tokens": response.usage_metadata.candidates_token_count if response.usage_metadata else None,
            "thinking_tokens": getattr(response.usage_metadata, 'thoughts_token_count', None),
        }
    }


## 6. Run the pilot

Writes each result to a JSONL file in Drive as it goes, so a crash mid-run doesn't lose work.


In [17]:
OUT_DIR = pathlib.Path("/content/drive/MyDrive/dissertation/pilot_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG = OUT_DIR / f"pilot_{datetime.now():%Y%m%d_%H%M%S}.jsonl"
print(f"Logging to {LOG}")

with LOG.open("w") as f:
    for gen_label, rel_path in PILOT_VIDEOS:
        video_path = f"{PILOT_ROOT}/{rel_path}"
        if not pathlib.Path(video_path).exists():
            print(f"MISSING: {video_path} — skipping")
            continue
        for prompt_name, prompt_text in PROMPTS.items():
            print(f"→ {gen_label:12s}  {pathlib.Path(rel_path).name:40s}  [{prompt_name}]", end=" ")
            t0 = time.time()
            try:
                result = analyse_video(video_path, prompt_text)
                record = {
                    "video_id": pathlib.Path(rel_path).stem,
                    "generator": gen_label,
                    "prompt_variant": prompt_name,
                    "elapsed_s": round(time.time() - t0, 1),
                    "raw_response": result["raw_text"],
                    "usage": result["usage"],
                }
                f.write(json.dumps(record) + "\n"); f.flush()
                print(f"OK ({record['elapsed_s']}s)")
            except Exception as e:
                print(f"FAIL: {e}")
                f.write(json.dumps({
                    "video_id": pathlib.Path(rel_path).stem,
                    "generator": gen_label,
                    "prompt_variant": prompt_name,
                    "error": str(e),
                }) + "\n"); f.flush()
            time.sleep(1)  # gentle rate-limit buffer

print(f"\nDone. Results at {LOG}")


Logging to /content/drive/MyDrive/dissertation/pilot_results/pilot_20260729_155200.jsonl
→ LTX           w2_005_ltx_20260719_105736.mp4            [A_baseline] OK (43.6s)
→ LTX           w2_005_ltx_20260719_105736.mp4            [B_calibrated] OK (33.5s)
→ LTX           w2_005_ltx_20260719_105736.mp4            [C_observe_first] OK (33.8s)
→ LTX           w2_020_ltx_20260719_110441.mp4            [A_baseline] OK (34.8s)
→ LTX           w2_020_ltx_20260719_110441.mp4            [B_calibrated] OK (31.8s)
→ LTX           w2_020_ltx_20260719_110441.mp4            [C_observe_first] OK (26.0s)
→ Hunyuan       w2_003_hunyuan_20260719_133334.mp4        [A_baseline] OK (21.6s)
→ Hunyuan       w2_003_hunyuan_20260719_133334.mp4        [B_calibrated] OK (40.0s)
→ Hunyuan       w2_003_hunyuan_20260719_133334.mp4        [C_observe_first] OK (25.0s)
→ Hunyuan       w2_015_hunyuan_20260719_141252.mp4        [A_baseline] OK (28.4s)
→ Hunyuan       w2_015_hunyuan_20260719_141252.mp4        [B_calibrate

## 7. Parse results and score the variants

In [18]:
import pandas as pd
from statistics import stdev, mean

FAMILIES = ["1.1_texture", "1.2_boundary", "1.3_lighting", "1.4_watermark",
            "2.1_human_anatomy", "2.2_non_human_anatomy", "2.3_object_structural",
            "2.4_scene_composition", "3.1_motion", "3.2_identity_drift",
            "3.3_continuity", "3.4_causality"]

def extract_json(text):
    """Return a dict, or None. Handles list-wrapped, markdown-fenced, or preamble-prefixed output."""
    if not text or not isinstance(text, str):
        return None
    # Strip Variant C observation preamble if present
    if "OBSERVATION:" in text and "{" in text:
        text = text[text.index("{"):]
    # Strip markdown fences
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1] if "\n" in t else t
        if t.endswith("```"):
            t = t.rsplit("```", 1)[0]
    t = t.strip()
    # Try direct parse
    obj = None
    try:
        obj = json.loads(t)
    except Exception:
        # Try to extract the first {...} block
        m = re.search(r"\{.*\}", t, re.DOTALL)
        if m:
            try:
                obj = json.loads(m.group(0))
            except Exception:
                return None
        else:
            return None
    # Unwrap list if the model returned [{...}]
    if isinstance(obj, list):
        obj = obj[0] if obj else None
    # Only accept dicts that have a "families" key
    if not isinstance(obj, dict) or "families" not in obj:
        return None
    return obj

# Load and diagnose
records = [json.loads(l) for l in LOG.read_text().splitlines() if l.strip()]
print(f"Loaded {len(records)} raw records")

# Diagnostic: how many have raw_response vs error?
with_response = [r for r in records if "raw_response" in r]
with_error    = [r for r in records if "error" in r]
print(f"  with response: {len(with_response)}")
print(f"  with error:    {len(with_error)}")
if with_error:
    print("  Error samples:")
    for r in with_error[:3]:
        print(f"    [{r.get('prompt_variant')}] {r.get('video_id')}: {r.get('error')[:120]}")

# Per-variant counts
from collections import Counter
by_variant = Counter(r.get("prompt_variant") for r in with_response)
print(f"  Responses per variant: {dict(by_variant)}")

scores = {}
for variant in ["A_baseline", "B_calibrated", "C_observe_first"]:
    v_recs = [r for r in with_response if r.get("prompt_variant") == variant]
    if not v_recs:
        print(f"\nNo responses for {variant} — skipping")
        continue

    parsed = [extract_json(r["raw_response"]) for r in v_recs]
    valid = [p for p in parsed if p is not None]

    # 1. JSON validity
    validity = len(valid) / len(v_recs)

    # 2. Schema completeness — guard every access
    def has_all_families(p):
        fams = p.get("families") if isinstance(p, dict) else None
        return isinstance(fams, dict) and all(f in fams for f in FAMILIES)
    complete = sum(1 for p in valid if has_all_families(p))
    schema_ok = complete / len(valid) if valid else 0
    has_14   = sum(1 for p in valid if isinstance(p.get("families"), dict)
                   and "1.4_watermark" in p["families"]) / len(valid) if valid else 0

    # 3. Severity distribution
    all_sevs = []
    for p in valid:
        fams = p.get("families", {}) if isinstance(p, dict) else {}
        if not isinstance(fams, dict):
            continue
        for f in FAMILIES:
            fam = fams.get(f)
            if isinstance(fam, dict) and fam.get("detected"):
                s = fam.get("severity")
                if s in ("L", "M", "H"):
                    all_sevs.append(s)
    if all_sevs:
        n = len(all_sevs)
        sev_dist = {s: all_sevs.count(s) / n for s in ("L", "M", "H")}
    else:
        sev_dist = {"L": 0, "M": 0, "H": 0}

    # 4. Pexels false-positive
    pex_pairs = [(r, p) for r, p in zip(v_recs, parsed)
                 if r.get("generator") == "Pexels" and isinstance(p, dict)]
    if pex_pairs:
        fp = sum(1 for _, p in pex_pairs if p.get("video_verdict") == "AI-generated")
        fp_rate = fp / len(pex_pairs)
        pex_wm = sum(1 for _, p in pex_pairs
                     if isinstance(p.get("families"), dict)
                     and isinstance(p["families"].get("1.4_watermark"), dict)
                     and p["families"]["1.4_watermark"].get("detected"))
        pex_wm_rate = pex_wm / len(pex_pairs)
    else:
        fp_rate = None
        pex_wm_rate = None

    # 5. Confidence spread
    confs = [p.get("confidence") for p in valid
             if isinstance(p.get("confidence"), (int, float))]
    conf_sd   = round(stdev(confs), 1) if len(confs) > 1 else 0
    conf_mean = round(mean(confs), 1) if confs else None

    # 6. Evidence length
    ev_lens = []
    for p in valid:
        fams = p.get("families", {}) if isinstance(p, dict) else {}
        for f in FAMILIES:
            fam = fams.get(f) if isinstance(fams, dict) else None
            if isinstance(fam, dict) and fam.get("detected") and fam.get("evidence"):
                ev_lens.append(len(fam["evidence"]))
    ev_mean = round(mean(ev_lens), 1) if ev_lens else 0

    scores[variant] = {
        "n_calls":              len(v_recs),
        "json_valid_%":         round(validity * 100, 1),
        "all_12_families_%":    round(schema_ok * 100, 1),
        "1.4_present_%":        round(has_14 * 100, 1),
        "sev_L_%":              round(sev_dist["L"] * 100, 1),
        "sev_M_%":              round(sev_dist["M"] * 100, 1),
        "sev_H_%":              round(sev_dist["H"] * 100, 1),
        "pexels_false_pos_%":   round(fp_rate * 100, 1) if fp_rate is not None else None,
        "pexels_watermark_fp_%": round(pex_wm_rate * 100, 1) if pex_wm_rate is not None else None,
        "confidence_mean":      conf_mean,
        "confidence_sd":        conf_sd,
        "evidence_chars_mean":  ev_mean,
    }

if scores:
    df = pd.DataFrame(scores).T
    df.to_csv(OUT_DIR / f"pilot_scores_{datetime.now():%Y%m%d_%H%M%S}.csv")
    print("\n" + df.to_string())
else:
    print("\nNo scores computed — check the diagnostics above.")

# Also show a sample raw response so you can see what shape Gemini is returning
if with_response:
    print("\n=== Sample raw response (first record) ===")
    print(with_response[0].get("raw_response", "")[:800])

Loaded 30 raw records
  with response: 30
  with error:    0
  Responses per variant: {'A_baseline': 10, 'B_calibrated': 10, 'C_observe_first': 10}

                 n_calls  json_valid_%  all_12_families_%  1.4_present_%  sev_L_%  sev_M_%  sev_H_%  pexels_false_pos_%  pexels_watermark_fp_%  confidence_mean  confidence_sd  evidence_chars_mean
A_baseline          10.0          80.0              100.0          100.0      6.7     93.3      0.0                 NaN                    NaN             70.4            2.0                 30.4
B_calibrated        10.0          90.0              100.0          100.0     11.5     88.5      0.0                 NaN                    NaN             73.3            2.5                 42.9
C_observe_first     10.0          90.0              100.0          100.0      1.8     98.2      0.0                 NaN                    NaN             70.0            2.5                 33.7

=== Sample raw response (first record) ===
{
  "video_verdict": "A

## 8. Decision rubric

Look at the table above and pick the variant that:

1. **False-positive rate on Pexels = 0%.** This is a hard filter — any variant that calls real videos AI-generated is unusable regardless of other metrics.
2. **Watermark false-positive on Pexels = 0%.** Same reason.
3. **Severity M-share ≤ 70%** (the old prompt was ~90%). Meaningful use of L and H.
4. **Confidence SD ≥ 10.** Old data clustered at 60–75 (SD ~5). We want the model actually distinguishing certainty.
5. **All 12 families present in 100% of responses**, especially 1.4.

If (1) and (2) tie, prefer the variant with better (3), (4), (5). Report the winning variant in your methodology chapter as "selected via pilot on n=14 videos"; report all three variants' scores as an appendix table for transparency.
